In [ ]:
import pandas as pd
import numpy as np
import statsmodels.api as sm

In [ ]:
df = pd.read_csv("../../output/processed_data/03_aspi_handled.csv")

# Convert to correct types
df['date'] = pd.to_datetime(df['date'])

# Important numeric columns
df['close'] = pd.to_numeric(df['close'], errors='coerce')
df['aspi_close'] = pd.to_numeric(df['aspi_close'], errors='coerce')

# Sort properly
df = df.sort_values(['symbol', 'date'])

In [ ]:
df.groupby('symbol')[['symbol', 'date']].head()

In [ ]:
# Stock return
df['R_i'] = df.groupby('symbol')['close'].pct_change()

# Market return (same for all stocks per day)
df['R_m'] = df.groupby('symbol')['aspi_close'].pct_change()

In [ ]:
event_date = pd.to_datetime("2025-11-28")

df['day'] = (df['date'] - event_date).dt.days

# Estimation window
estimation_df = df[(df['day'] >= -120) & (df['day'] <= -6)]

In [ ]:
results = []

for symbol, group in estimation_df.groupby('symbol'):

    # Drop NaNs
    group = group[['R_i', 'R_m']].dropna()

    if len(group) < 30:  # safety check
        continue

    X = group['R_m']
    y = group['R_i']

    # Add constant (for alpha)
    X = sm.add_constant(X)

    model = sm.OLS(y, X).fit()

    alpha = model.params['const']
    beta = model.params['R_m']

    results.append({
        'symbol': symbol,
        'alpha': alpha,
        'beta': beta
    })

alpha_beta_df = pd.DataFrame(results)

In [ ]:
alpha_beta_df.head()

In [ ]:
df = df.merge(alpha_beta_df, on='symbol', how='left')
df.head()

In [ ]:
df.to_csv('../../output/processed_data/04_Regression_handled.csv',index = False)